In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import json
from itertools import islice
from river import metrics, tree, drift
 
from ensemble import DriftAdaptiveEnsemble
from centroid_drift import CentroidDriftDetector
from transformer import FeatureDistrict
from river import metrics, dummy, forest, tree, stats, ensemble, drift
from utils import warsaw_stream, airline_stream, taxi_stream
from river.forest import ARFRegressor
from river.tree import HoeffdingTreeRegressor
from river.drift import ADWIN, PageHinkley
from river import metrics
import copy
from collections import defaultdict
from itertools import islice

import os

In [2]:
# %pip install -r requirements.txt

In [3]:
DATASET = "Synthetic_airplanes"

In [4]:
df_warsaw=pd.read_csv("dataset/warsaw_synthetic.csv")
df_airplanes=pd.read_csv("dataset/Airplanes_modified.csv")
df_synthetic_airplanes=pd.read_csv("dataset/Airplanes_synthetic.csv")
df_synthetic_airplanes=df_synthetic_airplanes.drop(columns=["region"])
df_taxi=pd.read_csv("dataset/taxi_dataset_ordered.csv")

if DATASET=="Airplanes":
    rows = df_airplanes.to_dict(orient='records')
    data = airline_stream(rows)
    leng=len(df_airplanes)
    size=df_airplanes.size()
elif DATASET=="Warsaw":
    rows = df_warsaw.to_dict(orient='records')
    data = warsaw_stream(rows)
    leng=df_warsaw.shape[0]
    size=df_warsaw.shape[1]
elif DATASET=="Taxi":
    rows = df_taxi.to_dict(orient='records')
    data = taxi_stream(rows)
    leng=df_taxi.shape[0]
    size=df_taxi.shape[1]
elif DATASET=="Synthetic_airplanes":
    rows = df_synthetic_airplanes.to_dict(orient='records')
    data = airline_stream(rows)
    leng=df_synthetic_airplanes.shape[0]//2
    size=df_synthetic_airplanes.shape[1]
else:
    raise Exception("Dataset ERROR")
print(leng)
print(size)

1159560
17


In [5]:
df_synthetic_airplanes.head(10)

,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,CRSArrTime,CRSElapsedTime,ArrDelay,Distance,previous_dep_delay,scheduled_turnaround,origin_state,origin_lat,origin_long,dest_state,dest_lat,dest_long
0,1,1,2,2400.0,10,737,267.0,-3.0,1979,82.0,248.0,CA,33.942536,-118.408074,MI,42.212059,-83.348836
1,1,1,2,5.0,15,823,308.0,-19.0,2521,7.0,980.0,CA,38.695422,-121.590767,NY,40.639751,-73.778926
2,1,1,2,19.0,25,709,284.0,-24.0,2153,0.0,1440.0,AZ,33.434167,-112.008056,NY,40.639751,-73.778926
3,1,1,2,45.0,25,535,190.0,12.0,1431,0.0,1440.0,CA,38.695422,-121.590767,TX,32.895951,-97.037200
4,1,1,2,26.0,30,444,194.0,24.0,1449,0.0,1440.0,AK,61.174320,-149.996186,WA,47.448982,-122.309313
5,1,1,2,29.0,30,825,295.0,-27.0,2248,0.0,1440.0,NV,36.080361,-115.152333,NY,40.639751,-73.778926
6,1,1,2,33.0,30,605,215.0,11.0,1536,0.0,1440.0,CA,33.942536,-118.408074,MN,44.880547,-93.216922
7,1,1,2,21.0,30,831,301.0,-23.0,2430,0.0,1440.0,CA,34.056000,-117.601194,NY,40.639751,-73.778926
8,1,1,2,33.0,35,603,208.0,-12.0,1635,0.0,1440.0,CA,37.619002,-122.374843,TX,29.980472,-95.339722
9,1,1,2,30.0,35,420,165.0,-15.0,1189,0.0,1440.0,FL,28.428889,-81.316028,PR,18.439417,-66.001833


In [6]:
MAX_SAMPLES = leng
seed = 17
 
with open("transformer/config_dataset.json", "r", encoding="utf-8-sig") as f:
    config = json.load(f)
cfg = config[DATASET]

In [7]:
districts = cfg["district_list"]

columns_to_drop = cfg["columns_to_drop"]

WARMUP_DAYS = cfg["WARMUP_DAYS"]
PH_THRESHOLD = cfg["PH_THRESHOLD"]
PH_MIN_INSTANCES = cfg["PH_MIN_INSTANCES"]
PH_ALPHA = cfg["PH_ALPHA"]

In [8]:
DISTRICTS_SETTINGS=cfg["DISTRICTS_SETTINGS"]

In [9]:
transformer = FeatureDistrict(
    dataset=DATASET,
    columns_to_drop=cfg["columns_to_drop"],
)

In [10]:
data = islice(data, MAX_SAMPLES)

In [11]:
def make_models(districts, district_settings, base_estimator, metric, **ensemble_kwargs):
    return {
        district: DriftAdaptiveEnsemble(
            base_estimator=copy.deepcopy(base_estimator),
            drift_detector= drift.ADWIN(delta=DISTRICTS_SETTINGS[district]["delta_c"], clock=DISTRICTS_SETTINGS[district]["window"]),
            warning_detector=drift.ADWIN(delta=DISTRICTS_SETTINGS[district]["delta_w"]),
            metric=copy.deepcopy(metric),
            **ensemble_kwargs,
        )
        for district in districts
    }





In [12]:
# DISTRICTS_SETTINGS = {
#     "South": {"delta_w": 0.7, "delta_c": 0.5, "window": 2000},    
#     "West": {"delta_w": 0.7, "delta_c": 0.5, "window": 2000},         
#     "Midwest": {"delta_w": 0.7, "delta_c":  0.5, "window": 300},    
#     "Northeast": {"delta_w": 0.5, "delta_c": 0.2, "window": 200},    
#     "Global": {"delta_w": 0.5, "delta_c": 0.1, "window": 1000}      
# }

In [13]:
rmse_metric = metrics.RMSE()

feature_drift_detectors = {d: CentroidDriftDetector(
    warmup_days=WARMUP_DAYS,
    ph_threshold=PH_THRESHOLD,
    ph_min_instances=PH_MIN_INSTANCES,
    ph_alpha=PH_ALPHA,
) for d in districts}

In [14]:
#m=HoeffdingTreeRegressor()
m=ensemble.SRPRegressor(n_models=2, seed=17)
models = make_models(districts, DISTRICTS_SETTINGS, m, metrics.RMSE(), max_ensemble_size=4, retain_initial_model=True)


In [15]:
records = []
drift_records = []
event_offsets = {k: 0 for k in models.keys()}
local_counts  = {d: 0 for d in districts}

feature_drift_records=[]

In [16]:
for i, (x_raw, y) in enumerate(tqdm(data, total=MAX_SAMPLES)):
    
    x = transformer.transform_one(x_raw)
    timestamp = x['timestamp']
    x.pop("timestamp")
    district = (
            x["pickup_district"]
            if x["within_district"] == 1
            else "Global"
        )
    x.pop("pickup_district", None)
    x.pop("dropoff_district", None)
    local_counts[district] += 1
    instance_count = local_counts[district]

    model = models[district]
    y_hat = model.predict_one(x)
    model.learn_one(x, y, timestamp)

    records.append({
            "n_seen":         i,
            "timestamp": timestamp,
            "district":  district,
            "y":         y,
            "y_hat":     y_hat,
        })

    
    log = model.drift_log
    offset = event_offsets[district]
    if len(log) > offset:
        for _, event in log.iloc[offset:].iterrows():
            drift_records.append({**event, "district": district})
        event_offsets[district] = len(log)
 
 
    feature_vec = np.array(list(x.values()), dtype=float)
    feature_drift_detectors[district].update(feature_vec, timestamp, instance_count)
 
    if feature_drift_detectors[district].drift_detected:
        last = feature_drift_detectors[district].drift_log[-1]
        last["district"] = district
        last["event_type"]="centroid"
        feature_drift_records.append(last)
        

100%|██████████| 1159560/1159560 [59:49<00:00, 323.03it/s] 


In [17]:
df_events=pd.DataFrame(drift_records)
df_events_feature=pd.DataFrame(feature_drift_records)

df_events.to_csv(f"results_drift/results_raw/{DATASET}_events.csv", index=False)
df_events_feature.to_csv(f"results_drift/results_raw/{DATASET}_events_centroid.csv", index=False)

df_predictions = pd.DataFrame(records)


In [18]:
df_predictions.tail(5)

,n_seen,timestamp,district,y,y_hat
1159555,1159555,2008-03-02 13:00:00,South,9.0,-0.419449
1159556,1159556,2008-03-02 13:00:00,Global,28.0,23.340793
1159557,1159557,2008-03-02 13:00:00,South,23.0,2.600289
1159558,1159558,2008-03-02 13:00:00,South,-14.0,-0.198369
1159559,1159559,2008-03-02 13:00:00,Northeast,-19.0,10.813881


In [19]:
feature_drift_records

[{'date': Timestamp('2008-02-07 00:00:00'),
  'instance': 14531,
  'distance': 0.5342577402410464,
  'daily_centroid': array([ 1.32632009e+03,  1.42856776e+03,  8.26214953e+01,  2.35892523e+02,
          7.41121495e+00,  3.56782710e+02,  4.14593759e+01, -7.39171992e+01,
          4.14674291e+01, -7.38646194e+01,  0.00000000e+00,  1.00000000e+00]),
  'district': 'Northeast',
  'event_type': 'centroid'},
 {'date': Timestamp('2008-02-10 00:00:00'),
  'instance': 169613,
  'distance': 2.124662247392121,
  'daily_centroid': array([ 1.35598444e+03,  1.50801556e+03,  9.82650134e+01,  4.88037442e+02,
          3.72625821e+01,  3.59345247e+02,  3.27981506e+01, -8.77818174e+01,
          3.27982870e+01, -8.77672627e+01,  1.00000000e+00,  1.00000000e+00]),
  'district': 'South',
  'event_type': 'centroid'},
 {'date': Timestamp('2008-02-15 00:00:00'),
  'instance': 54810,
  'distance': 0.5679762548999917,
  'daily_centroid': array([ 1.34720254e+03,  1.46802167e+03,  7.62002990e+01,  2.90484305e+02

In [20]:
df_events_2 = df_events[
    (df_events["event_type"] == "drift") |
    (df_events["event_type"] == "warning")
]
df_events_2=df_events_2[["timestamp","n_seen","event_type","district"]]
df_centroid=df_events_feature[["date","instance","event_type","district"]]
df_centroid = df_centroid.rename(columns={
    "date": "timestamp",
    "instance": "n_seen"
})
drift_log_final=pd.concat([df_events_2, df_centroid], axis=0)
drift_log_final.tail(5)


,timestamp,n_seen,event_type,district
2,2008-02-15,54810,centroid,Midwest
3,2008-02-17,18050,centroid,Northeast
4,2008-02-21,61888,centroid,Midwest
5,2008-02-24,228226,centroid,South
6,2008-02-26,21295,centroid,Northeast


In [21]:
df_predictions.to_csv(f"results_drift/{DATASET}_predictions.csv", index=False)
drift_log_final.to_csv(f"results_drift/{DATASET}_drift_log.csv", index=False)


In [22]:

# drift_centroid = pd.DataFrame(feature_drift_detectors[district].drift_log)

# drift_centroid2 = drift_centroid.drop(columns=["daily_centroid"], errors="ignore")

# drift_centroid2.to_csv(f"{DATASET}_centroid_drift_log.csv", index=False)